<a href="https://colab.research.google.com/github/helmernet/Helmernet/blob/main/GeneraVd01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
# -*- coding: utf-8 -*-
"""
Este script permite generar un video a partir de texto, múltiples imágenes o videos de fondo,
y opcionalmente, música de fondo. Está diseñado para ser ejecutado en Google Colab.
"""

# Instalar las librerías necesarias si aún no están instaladas
!pip install gtts moviepy imageio-ffmpeg

import os
from moviepy.editor import *
from gtts import gTTS
from google.colab import files
import math

# Duración de cada segmento de imagen/video en el fondo
SEGMENT_DURATION = 6 # segundos
MAX_MEDIA_FILES = 12 # Número máximo de imágenes o videos permitidos

# Función para convertir texto a voz
def text_to_speech(text, lang='es', output_file="temp_audio.mp3"):
    """
    Convierte el texto dado a voz utilizando gTTS y lo guarda en un archivo MP3.

    Args:
        text (str): El texto a convertir a voz.
        lang (str): El idioma del texto (ej. 'es' para español, 'en' para inglés).
        output_file (str): El nombre del archivo de salida para el audio.

    Returns:
        str: La ruta del archivo de audio generado.
    """
    tts = gTTS(text=text, lang=lang)
    tts.save(output_file)
    return output_file

# Función para aplicar el efecto Ken Burns (zoom lento) a un clip
def apply_ken_burns_effect(clip, segment_duration, final_video_width, final_video_height):
    """
    Aplica un efecto de zoom lento (Ken Burns) a un clip de imagen o video.

    Args:
        clip (ImageClip o VideoFileClip): El clip al que se le aplicará el efecto.
        segment_duration (int): La duración del segmento en segundos.
        final_video_width (int): Ancho del video final.
        final_video_height (int): Altura del video final.

    Returns:
        VideoClip: El clip con el efecto de Ken Burns aplicado.
    """
    start_zoom_factor = 1.0  # La imagen/video comienza a su tamaño inicial (adaptado al marco)
    end_zoom_factor = 1.15   # La imagen/video termina un 15% más grande

    # Función lambda para calcular el factor de escala en un momento dado 't' dentro del segmento
    # 't' va de 0 a segment_duration
    get_current_zoom_scale = lambda t: start_zoom_factor + (end_zoom_factor - start_zoom_factor) * (t / segment_duration)

    # Calcular las dimensiones iniciales para que el clip "cubra" el marco de video
    clip_aspect_ratio = clip.w / clip.h
    video_aspect_ratio = final_video_width / final_video_height

    if clip_aspect_ratio > video_aspect_ratio:
        # El clip es más ancho que el frame del video, lo escalamos por altura
        initial_scaled_height = final_video_height
        initial_scaled_width = int(final_video_height * clip_aspect_ratio)
    else:
        # El clip es más alto o igual, lo escalamos por ancho
        initial_scaled_width = final_video_width
        initial_scaled_height = int(final_video_width / clip_aspect_ratio)

    # Aplicar el zoom animado al clip
    animated_clip = clip.resize(
        width=lambda t: initial_scaled_width * get_current_zoom_scale(t),
        height=lambda t: initial_scaled_height * get_current_zoom_scale(t)
    ).set_position(("center", "center")) # Mantener el centro para un zoom centrado

    # Crear un clip de fondo negro del tamaño final del video
    black_background_clip = ColorClip(size=(final_video_width, final_video_height), color=(0,0,0), duration=segment_duration)

    # Componer el video final con el fondo negro y el clip animado encima
    final_segment_clip = CompositeVideoClip([black_background_clip, animated_clip])
    return final_segment_clip.set_duration(segment_duration) # Asegurar la duración del segmento

# Función para crear el video final
def create_video_with_music(text, media_paths, media_type, audio_file, music_file=None, output_video="output_video.mp4"):
    """
    Crea un video combinando múltiples clips de medios (imágenes o videos),
    un clip de audio de voz y opcionalmente un clip de música de fondo.
    Aplica efecto de zoom a cada segmento.

    Args:
        text (str): El texto que se usó para generar el audio (no se usa directamente aquí).
        media_paths (list): Lista de rutas a los archivos de imagen o video.
        media_type (str): '1' para imágenes, '2' para videos.
        audio_file (str): La ruta al archivo de audio de voz.
        music_file (str, opcional): La ruta al archivo de música de fondo. Por defecto es None.
        output_video (str): El nombre del archivo de salida para el video final.

    Returns:
        str: La ruta del archivo de video generado.
    """
    if not media_paths:
        raise ValueError("No se han proporcionado archivos de medios para crear el video.")

    # Definir el tamaño del video de salida (ej. 720p para HD)
    final_video_height = 720
    final_video_width = int(final_video_height * (16 / 9)) # Asumiendo una relación de aspecto 16:9

    # Calcular la duración total del video basada en el número de segmentos
    total_video_duration = len(media_paths) * SEGMENT_DURATION

    video_segments = []

    for i, path in enumerate(media_paths):
        print(f"Procesando medio {i+1}/{len(media_paths)}: {os.path.basename(path)}")
        if media_type == '2': # Es un video
            base_clip = VideoFileClip(path)
            # Asegurarse de que el clip de video sea al menos tan largo como el segmento
            if base_clip.duration < SEGMENT_DURATION:
                # Si el video es más corto, lo duplica hasta que dure el segmento
                base_clip = base_clip.loop(duration=SEGMENT_DURATION)
            # Recortar el clip a la duración del segmento
            segment_clip = base_clip.subclip(0, SEGMENT_DURATION)
            # Aplicar efecto de zoom al video
            processed_segment = apply_ken_burns_effect(segment_clip, SEGMENT_DURATION, final_video_width, final_video_height)
        else: # Es una imagen
            base_clip = ImageClip(path)
            # Aplicar efecto de zoom a la imagen
            processed_segment = apply_ken_burns_effect(base_clip, SEGMENT_DURATION, final_video_width, final_video_height)

        video_segments.append(processed_segment)

    # Concatenar todos los segmentos de video
    final_media_clip = concatenate_videoclips(video_segments)

    # Cargar el archivo de audio (voz)
    # Si la duración del audio de voz es mayor que la duración total del video, se recortará.
    # Si es menor, el video podría terminar antes de que el audio se reproduzca completamente.
    audio_clip_full = AudioFileClip(audio_file)
    audio_clip = audio_clip_full.subclip(0, min(total_video_duration, audio_clip_full.duration))


    # Combinar media y audio
    video_clip = final_media_clip.set_audio(audio_clip)

    # Si se proporciona música de fondo, mezclarla con el audio
    if music_file:
        music_clip_full = AudioFileClip(music_file)
        # Asegurarse de que la música de fondo tenga la duración total del video (o menos si la canción es más corta)
        music_clip = music_clip_full.subclip(0, min(total_video_duration, music_clip_full.duration))
        music_clip = music_clip.volumex(0.2) # Bajar el volumen de la música

        # Mezclar el audio de voz con la música de fondo
        final_audio = CompositeAudioClip([audio_clip, music_clip])
        video_clip = video_clip.set_audio(final_audio)

    # Guardar el video final
    video_clip.write_videofile(output_video, fps=24, codec="libx264")
    return output_video

# Función principal para generar el video y permitir su descarga
def generate_and_download_video():
    """
    Función principal que guía al usuario a través del proceso de creación de video,
    desde la entrada de texto y la selección de múltiples medios hasta la descarga del video final.
    """
    print("¡Bienvenido al generador de video personalizado!")
    # Solicitar al usuario el texto para el video
    input_text = input("Ingresa el texto para el video: ")
    # Solicitar el idioma, con 'es' (español) como valor predeterminado
    lang = input("Selecciona el idioma (es/en, por defecto 'es'): ") or "es"

    # La duración máxima del audio (y por lo tanto del video) sigue siendo 120 segundos.
    # Esta duración limitará la cantidad de texto de voz que se procesará.
    while True:
        try:
            max_audio_duration = int(input(f"Ingresa la duración MÁXIMA deseada para la voz (segundos, máx. 120, por defecto 30): ") or 30)
            if 0 < max_audio_duration <= 120:
                break
            else:
                print("La duración debe ser entre 1 y 120 segundos.")
        except ValueError:
            print("Entrada inválida. Por favor, ingresa un número.")


    # Convertir texto a voz para la duración máxima permitida.
    # La duración real del audio que se use en el video dependerá de la duración total de los segmentos de medios.
    audio_file = text_to_speech(input_text, lang=lang, output_file="temp_audio.mp3")

    # Cargar el audio de voz para obtener su duración real y poder ajustarla al final
    voice_audio_duration = AudioFileClip(audio_file).duration

    # --- INICIO DE LA SECCIÓN DE CARGA DE MEDIOS ---
    print("\n--- CARGA DE FONDOS (IMÁGENES O VIDEOS) ---")
    print(f"Puedes subir hasta {MAX_MEDIA_FILES} archivos. Cada uno se mostrará por {SEGMENT_DURATION} segundos con efecto de movimiento.")
    print("Por favor, selecciona TODOS los archivos de fondo (imágenes o videos) que deseas usar en un solo paso.")
    print("  - Si eliges IMÁGENES, selecciona archivos JPG/PNG.")
    print("  - Si eliges VIDEOS, selecciona archivos MP4/MOV/AVI.")

    uploaded_media_dict = files.upload()

    media_paths = list(uploaded_media_dict.keys())

    if not media_paths:
        print("No se subió ningún archivo de fondo. El video no se generará.")
        # Limpiar audio generado si no se va a usar el video
        if os.path.exists(audio_file):
            os.remove(audio_file)
        return # Salir de la función si no hay medios

    if len(media_paths) > MAX_MEDIA_FILES:
        print(f"Se han subido más de {MAX_MEDIA_FILES} archivos. Se usarán los primeros {MAX_MEDIA_FILES}.")
        media_paths = media_paths[:MAX_MEDIA_FILES]

    # Determinar automáticamente el tipo de medio basado en la primera extensión
    # Esto asume que el usuario subirá solo imágenes O solo videos en una carga.
    first_file_extension = os.path.splitext(media_paths[0])[1].lower()
    if first_file_extension in ['.mp4', '.mov', '.avi', '.mkv']:
        media_type = '2' # Es un video
        print(f"Detectado: Cargando {len(media_paths)} videos como fondo.")
    else:
        media_type = '1' # Es una imagen
        print(f"Detectado: Cargando {len(media_paths)} imágenes como fondo.")

    # Ajustar la duración total del video para que coincida con la duración de la voz
    # o la duración máxima de los segmentos, lo que sea menor.
    total_segments_duration = len(media_paths) * SEGMENT_DURATION
    final_video_duration = min(total_segments_duration, voice_audio_duration, max_audio_duration)

    if final_video_duration < total_segments_duration:
        print(f"La duración de la voz ({voice_audio_duration:.1f}s) es más corta que la duración total de los segmentos ({total_segments_duration:.1f}s).")
        print(f"El video final durará {final_video_duration:.1f} segundos y puede que no muestre todos los segmentos.")
        # Opcionalmente, podrías recortar `media_paths` aquí para que no exceda la duración de la voz
        # Por ahora, simplemente moviepy cortará el video a la duración del audio si este es más corto.
    elif final_video_duration > total_segments_duration:
        print(f"La duración de la voz ({voice_audio_duration:.1f}s) es más larga que la duración de los segmentos de fondo ({total_segments_duration:.1f}s).")
        print(f"El video final durará {total_segments_duration:.1f} segundos y la voz se recortará.")
        final_video_duration = total_segments_duration # Ajustar para que no haya silencio sin video

    # --- FIN DE LA SECCIÓN DE CARGA DE MEDIOS ---

    # Permitir al usuario subir un archivo de música de fondo (opcional)
    print("\n--- CARGA DE MÚSICA DE FONDO (Opcional) ---")
    print("Sube un archivo de música (MP3, WAV) si deseas. Será mezclado con la voz.")
    uploaded_music = files.upload()
    music_file = list(uploaded_music.keys())[0] if uploaded_music else None

    # Crear el video
    print("\nGenerando video... Esto puede tardar varios minutos dependiendo de la cantidad y tipo de archivos.")
    output_video = create_video_with_music(
        input_text, # text no se usa en create_video_with_music, pero se mantiene por compatibilidad
        media_paths=media_paths,
        media_type=media_type,
        audio_file=audio_file,
        music_file=music_file,
        output_video="output_video.mp4"
    )

    print(f"\n¡Video '{output_video}' generado con éxito!")
    # Permitir la descarga del video
    files.download(output_video)

    # Limpieza de archivos temporales
    print("\nLimpiando archivos temporales...")
    if os.path.exists("temp_audio.mp3"):
        os.remove("temp_audio.mp3")
    for path in media_paths:
        if os.path.exists(path):
            os.remove(path)
    if music_file and os.path.exists(music_file):
        os.remove(music_file)
    print("Limpieza completada.")


# Ejecutar la función principal cuando el script se ejecuta directamente
if __name__ == "__main__":
    generate_and_download_video()

¡Bienvenido al generador de video personalizado!
Ingresa el texto para el video: Cansado de proyectos con retrasos y sobrecostos. Busca soluciones de ingeniería estructural confiable. En WAPB Ingeniería Estructural convertimos desafíos en soluciones innovadoras.  Somos WAPB Ingeniería Estructural una empresa líder en el diseño y desarrollo de soluciones de ingeniería para proyectos de infraestructura.  Ofrecemos un servicio integral desde la concepción hasta la ejecución  garantizando calidad y seguridad Nuestra experiencia abarca los sectores civil, eléctrico y energético. Ofrecemos una amplia gama de servicios incluyendo diseños y construcción de proyectos con metodología BIM.
Selecciona el idioma (es/en, por defecto 'es'): es
Ingresa la duración MÁXIMA deseada para la voz (segundos, máx. 120, por defecto 30): 45

--- CARGA DE FONDOS (IMÁGENES O VIDEOS) ---
Puedes subir hasta 12 archivos. Cada uno se mostrará por 6 segundos con efecto de movimiento.
Por favor, selecciona TODOS los ar

Saving WAPB (1).JPG to WAPB (1).JPG
Saving WAPB (2).JPG to WAPB (2).JPG
Saving WAPB (3).JPG to WAPB (3).JPG
Saving WAPB (4).JPG to WAPB (4).JPG
Saving WAPB (5).JPG to WAPB (5).JPG
Saving WAPB (7).JPG to WAPB (7).JPG
Saving WAPB (8).JPG to WAPB (8).JPG
Detectado: Cargando 7 imágenes como fondo.

--- CARGA DE MÚSICA DE FONDO (Opcional) ---
Sube un archivo de música (MP3, WAV) si deseas. Será mezclado con la voz.


Saving [SPOTIFY-DOWNLOADER.COM] A Day In The Life.mp3 to [SPOTIFY-DOWNLOADER.COM] A Day In The Life.mp3

Generando video... Esto puede tardar varios minutos dependiendo de la cantidad y tipo de archivos.
Procesando medio 1/7: WAPB (1).JPG
Procesando medio 2/7: WAPB (2).JPG
Procesando medio 3/7: WAPB (3).JPG
Procesando medio 4/7: WAPB (4).JPG
Procesando medio 5/7: WAPB (5).JPG
Procesando medio 6/7: WAPB (7).JPG
Procesando medio 7/7: WAPB (8).JPG
Moviepy - Building video output_video.mp4.
MoviePy - Writing audio in output_videoTEMP_MPY_wvf_snd.mp3


MoviePy - Done.
Moviepy - Writing video output_video.mp4



Moviepy - Done !
Moviepy - video ready output_video.mp4

¡Video 'output_video.mp4' generado con éxito!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Limpiando archivos temporales...
Limpieza completada.
